# 07. 회귀분석 + Threshold 검증

이 노트북은 raw image/mask에서 추출된 component feature CSV를 입력으로 받아, `group` 기준으로 미세스크래치 제거 기준을 검증한다.

- `group` 문자열에 `미세스크래치`가 포함된 행: 필터링되어야 하는 대상, positive class
- `group` 문자열에 `미세스크래치`가 없는 행: 필터링되면 안 되는 대상, negative class

binary target이므로 여기서의 회귀분석은 Logistic Regression을 기본으로 한다. 이후 상위 feature를 대상으로 threshold sweep을 수행해, threshold별 미세스크래치 제거율과 다른 불량 오제거율을 확인한다.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from sklearn.impute import SimpleImputer
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
    from sklearn.model_selection import GroupShuffleSplit, train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except ImportError as exc:
    raise ImportError("이 노트북은 scikit-learn이 필요합니다. pip install scikit-learn 후 다시 실행하세요.") from exc

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True


## 07-1. 입력 설정

`CSV_PATH`에 실제 feature CSV 경로를 넣는다. 비워두면 `runs/` 아래의 `*.csv` 중 가장 최근 파일을 임시로 사용한다.

In [ ]:
ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "Vision" / "Process" / "Scratch_Postprocess",
    Path.cwd().parent,
]
PROJECT_ROOT = next((p for p in ROOT_CANDIDATES if (p / "scratch_postprocess_utils.py").exists()), Path.cwd())
RUNS_ROOT = PROJECT_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# TODO: 실제 CSV 경로를 여기에 입력한다.
CSV_PATH = ""

MICRO_KEYWORD = "미세스크래치"
RANDOM_STATE = 42
TEST_SIZE = 0.25
TOP_K_FEATURES = 8

# 미세스크래치 제거보다 다른 불량 오제거를 더 무겁게 벌점 처리한다.
MAX_FALSE_REMOVE_RATE = 0.05
FALSE_REMOVE_WEIGHT = 3.0

# absolute RGB mean은 조명/제품 색상에 과적합될 수 있으므로 기본 제외한다.
EXCLUDE_ABSOLUTE_COLOR_MEANS = True
MAX_MISSING_RATIO = 0.45
CORR_DROP_THRESHOLD = 0.95

def csv_has_group_column(path: Path) -> bool:
    for encoding in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            header = pd.read_csv(path, encoding=encoding, nrows=0)
            return "group" in header.columns
        except Exception:
            continue
    return False

if not CSV_PATH:
    csv_candidates = [p for p in RUNS_ROOT.glob("*.csv") if csv_has_group_column(p)]
    csv_candidates = sorted(csv_candidates, key=lambda p: p.stat().st_mtime)
    if not csv_candidates:
        raise FileNotFoundError("CSV_PATH에 group 컬럼이 있는 실제 feature CSV 경로를 입력해주세요.")
    CSV_PATH = str(csv_candidates[-1])

CSV_PATH = Path(CSV_PATH)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_PATH:", CSV_PATH)


In [ ]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    errors = []
    for encoding in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as exc:
            errors.append((encoding, str(exc)))
    raise RuntimeError(f"CSV를 읽지 못했습니다: {errors}")

raw_df = read_csv_flexible(CSV_PATH)
if "group" not in raw_df.columns:
    raise ValueError("CSV에 group 컬럼이 필요합니다.")

df = raw_df.copy()
df["target_filter"] = df["group"].astype(str).str.contains(MICRO_KEYWORD, na=False).astype(int)

print("rows:", len(df))
print("columns:", len(df.columns))
display(df.head())

target_summary = df.groupby("target_filter").size().rename(index={0: "keep_negative", 1: "remove_micro_positive"})
display(target_summary.to_frame("count"))
display(df.groupby(["group", "target_filter"]).size().reset_index(name="count").sort_values("count", ascending=False).head(50))


## 07-2. Feature Engineering

원본 feature를 그대로 쓰되, 회귀/threshold에 더 직접적인 형태를 몇 개 추가한다.

- `bbox_major`, `bbox_minor`: 방향에 덜 민감한 bounding box 크기
- `bbox_aspect_engineered`: 길쭉함
- `bbox_fill_ratio_engineered`: box 대비 component가 차지하는 비율
- `log_*`: area/contrast의 긴 꼬리 분포 완화
- `rgb_std_mean`, `rgb_std_max`: raw texture/noise 강도 요약

In [ ]:
ID_COLUMNS = {
    "image_path", "mask_path", "component_id", "group", "target_filter",
    "sample_id", "file_name", "filename", "path", "label"
}

ABSOLUTE_COLOR_MEAN_PREFIXES = (
    "perimeter_mean_", "center_mean_", "skeleton_mean_"
)

def first_existing(columns, candidates):
    for col in candidates:
        if col in columns:
            return col
    return None

def to_numeric_series(frame, col):
    return pd.to_numeric(frame[col], errors="coerce")

def build_modeling_frame(input_df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    work = input_df.copy()

    for col in work.columns:
        if col not in ID_COLUMNS:
            work[col] = pd.to_numeric(work[col], errors="ignore")

    area_col = first_existing(work.columns, ["area", "area_px"])
    bbox_w_col = first_existing(work.columns, ["bbox_width", "bbox_w"])
    bbox_h_col = first_existing(work.columns, ["bbox_height", "bbox_h"])

    if area_col is not None:
        area = to_numeric_series(work, area_col).clip(lower=0)
        work["log_area"] = np.log1p(area)

    if bbox_w_col is not None and bbox_h_col is not None:
        bw = to_numeric_series(work, bbox_w_col).clip(lower=0)
        bh = to_numeric_series(work, bbox_h_col).clip(lower=0)
        major = np.maximum(bw, bh)
        minor = np.minimum(bw, bh)
        work["bbox_major"] = major
        work["bbox_minor"] = minor
        work["bbox_aspect_engineered"] = major / (minor.replace(0, np.nan) + 1e-6)
        work["bbox_area_engineered"] = bw * bh
        work["log_bbox_major"] = np.log1p(major)
        work["log_bbox_minor"] = np.log1p(minor)
        if area_col is not None:
            work["bbox_fill_ratio_engineered"] = to_numeric_series(work, area_col) / (work["bbox_area_engineered"] + 1e-6)

    rgb_std_cols = [c for c in ["rgb_std_b", "rgb_std_g", "rgb_std_r"] if c in work.columns]
    if rgb_std_cols:
        std_values = work[rgb_std_cols].apply(pd.to_numeric, errors="coerce")
        work["rgb_std_mean"] = std_values.mean(axis=1)
        work["rgb_std_max"] = std_values.max(axis=1)

    for prefix in ["perimeter_center_diff", "perimeter_skeleton_diff"]:
        cols = [f"{prefix}_b", f"{prefix}_g", f"{prefix}_r"]
        existing = [c for c in cols if c in work.columns]
        for col in existing:
            work[f"abs_{col}"] = to_numeric_series(work, col).abs()
        if len(existing) == 3 and f"{prefix}_euclidean" not in work.columns:
            vals = work[existing].apply(pd.to_numeric, errors="coerce")
            work[f"{prefix}_euclidean"] = np.sqrt((vals ** 2).sum(axis=1))

    positive_magnitude_cols = [
        "luma_contrast_abs", "rgb_euclidean_contrast", "max_channel_contrast_abs", "mean_channel_contrast_abs",
        "perimeter_center_diff_euclidean", "perimeter_skeleton_diff_euclidean",
        "luma_contrast_z", "rgb_contrast_z"
    ]
    for col in positive_magnitude_cols:
        if col in work.columns:
            work[f"log_{col}"] = np.log1p(to_numeric_series(work, col).clip(lower=0))

    numeric_cols = [c for c in work.columns if pd.api.types.is_numeric_dtype(work[c])]
    feature_cols = []
    for col in numeric_cols:
        if col in ID_COLUMNS:
            continue
        if EXCLUDE_ABSOLUTE_COLOR_MEANS and col.startswith(ABSOLUTE_COLOR_MEAN_PREFIXES):
            continue
        feature_cols.append(col)

    feature_cols = [c for c in feature_cols if c != "target_filter"]
    return work, feature_cols

model_df, raw_feature_cols = build_modeling_frame(df)
print("raw numeric feature count:", len(raw_feature_cols))
display(model_df[raw_feature_cols].describe().T.head(80))


In [ ]:
def prepare_feature_matrix(frame: pd.DataFrame, feature_cols: list[str]):
    X = frame[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)

    missing_ratio = X.isna().mean()
    keep_cols = missing_ratio[missing_ratio <= MAX_MISSING_RATIO].index.tolist()
    X = X[keep_cols]

    nunique = X.nunique(dropna=True)
    keep_cols = nunique[nunique > 1].index.tolist()
    X = X[keep_cols]

    median_values = X.median(numeric_only=True).fillna(0)
    X_for_corr = X.fillna(median_values)
    corr = X_for_corr.corr(method="spearman").abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [col for col in upper.columns if any(upper[col] > CORR_DROP_THRESHOLD)]
    X = X.drop(columns=drop_cols)

    report = pd.DataFrame({
        "feature": feature_cols,
        "missing_ratio": missing_ratio.reindex(feature_cols).values,
        "nunique": nunique.reindex(feature_cols).values,
        "kept": [c in X.columns for c in feature_cols],
        "dropped_by_corr": [c in drop_cols for c in feature_cols],
    })
    return X, report, corr

y = model_df["target_filter"].astype(int)
if y.nunique() < 2:
    raise ValueError("target_filter가 한 class만 있습니다. group에 미세스크래치 포함/미포함 행이 모두 필요합니다.")

X, feature_prepare_report, feature_corr = prepare_feature_matrix(model_df, raw_feature_cols)

print("selected feature count:", X.shape[1])
display(feature_prepare_report.sort_values(["kept", "dropped_by_corr", "missing_ratio"], ascending=[True, False, False]).head(80))
display(X.head())


## 07-3. Train/Test Split

`image_path`가 있으면 이미지 단위로 train/test를 나눈다. 같은 이미지에서 나온 component가 train/test에 동시에 들어가면 실제보다 성능이 좋게 보일 수 있기 때문이다.

In [ ]:
def make_train_test_split(frame: pd.DataFrame, y: pd.Series):
    indices = np.arange(len(frame))
    if "image_path" in frame.columns and frame["image_path"].nunique(dropna=True) >= 4:
        groups = frame["image_path"].astype(str).fillna("unknown")
        for seed in range(RANDOM_STATE, RANDOM_STATE + 200):
            splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
            train_idx, test_idx = next(splitter.split(indices, y, groups))
            if y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2:
                return train_idx, test_idx, "GroupShuffleSplit(image_path)", seed
        print("image_path 기준 split에서 양쪽 class가 유지되지 않아 stratified split으로 fallback합니다.")

    if y.value_counts().min() < 2:
        raise ValueError("class별 행 수가 너무 적어 train/test split을 만들 수 없습니다.")
    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    return train_idx, test_idx, "Stratified random split", RANDOM_STATE

train_idx, test_idx, split_name, split_seed = make_train_test_split(model_df, y)
print(split_name, "seed=", split_seed)
print("train rows:", len(train_idx), "test rows:", len(test_idx))
display(pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_idx), len(test_idx)],
    "micro_positive": [int(y.iloc[train_idx].sum()), int(y.iloc[test_idx].sum())],
    "keep_negative": [int((1 - y.iloc[train_idx]).sum()), int((1 - y.iloc[test_idx]).sum())],
}))


## 07-4. Logistic Regression

계수는 standardization 이후의 값이므로 절대값이 클수록 target 구분에 강하게 기여한 feature로 해석한다. 다만 상관이 큰 feature는 계수가 흔들릴 수 있으므로, threshold 선정에는 univariate AUC도 함께 본다.

In [ ]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

def make_logistic_pipeline(penalty="elasticnet"):
    if penalty == "elasticnet":
        clf = LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=0.5,
            class_weight="balanced",
            max_iter=20000,
            random_state=RANDOM_STATE,
        )
    else:
        clf = LogisticRegression(
            penalty="l2",
            solver="liblinear",
            C=1.0,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", clf),
    ])

try:
    model = make_logistic_pipeline("elasticnet")
    model.fit(X_train, y_train)
    model_name = "LogisticRegression(elasticnet)"
except Exception as exc:
    print("elasticnet 실패, l2 logistic으로 fallback:", exc)
    model = make_logistic_pipeline("l2")
    model.fit(X_train, y_train)
    model_name = "LogisticRegression(l2)"

print("model:", model_name)


In [ ]:
def binary_filter_metrics(y_true, pred_remove, score=None):
    y_true = np.asarray(y_true).astype(int)
    pred_remove = np.asarray(pred_remove).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred_remove, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    micro_recall = tp / max(tp + fn, 1)
    false_remove_rate = fp / max(fp + tn, 1)
    keep_recall = tn / max(tn + fp, 1)
    out = {
        "rows": int(len(y_true)),
        "tp_micro_removed": int(tp),
        "fn_micro_kept": int(fn),
        "fp_nonmicro_removed": int(fp),
        "tn_nonmicro_kept": int(tn),
        "precision_micro_when_removed": precision,
        "micro_remove_recall": micro_recall,
        "false_remove_rate_nonmicro": false_remove_rate,
        "keep_recall_nonmicro": keep_recall,
    }
    if score is not None and len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, score)
        out["average_precision"] = average_precision_score(y_true, score)
    return out

prob_train = model.predict_proba(X_train)[:, 1]
prob_test = model.predict_proba(X_test)[:, 1]

model_eval = pd.DataFrame([
    {"split": "train", **binary_filter_metrics(y_train, prob_train >= 0.5, prob_train)},
    {"split": "test", **binary_filter_metrics(y_test, prob_test >= 0.5, prob_test)},
])
display(model_eval)


In [ ]:
coef = model.named_steps["clf"].coef_[0]
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef_standardized": coef,
    "abs_coef_standardized": np.abs(coef),
    "coef_direction": np.where(coef >= 0, "larger -> more micro", "smaller -> more micro"),
})

def univariate_auc_table(X_part: pd.DataFrame, y_part: pd.Series) -> pd.DataFrame:
    rows = []
    for col in X_part.columns:
        x = pd.to_numeric(X_part[col], errors="coerce")
        valid = x.notna()
        if valid.sum() < 4 or y_part[valid].nunique() < 2 or x[valid].nunique() < 2:
            rows.append({"feature": col, "univariate_auc": np.nan, "oriented_auc": np.nan, "threshold_direction_hint": "unknown"})
            continue
        auc = roc_auc_score(y_part[valid], x[valid])
        rows.append({
            "feature": col,
            "univariate_auc": auc,
            "oriented_auc": max(auc, 1 - auc),
            "threshold_direction_hint": "feature >= threshold" if auc >= 0.5 else "feature <= threshold",
        })
    return pd.DataFrame(rows)

uni_df = univariate_auc_table(X_train, y_train)
ranking = coef_df.merge(uni_df, on="feature", how="left")
coef_norm = ranking["abs_coef_standardized"] / max(ranking["abs_coef_standardized"].max(), 1e-9)
auc_norm = (ranking["oriented_auc"].fillna(0.5) - 0.5) / 0.5
ranking["combined_rank_score"] = 0.6 * coef_norm + 0.4 * auc_norm
ranking = ranking.sort_values("combined_rank_score", ascending=False).reset_index(drop=True)

ranking_path = RUNS_ROOT / "regression_feature_ranking.csv"
ranking.to_csv(ranking_path, index=False, encoding="utf-8-sig")
print("saved:", ranking_path)
display(ranking.head(30))


In [ ]:
top_plot = ranking.head(20).iloc[::-1]
plt.figure(figsize=(9, max(5, len(top_plot) * 0.28)))
plt.barh(top_plot["feature"], top_plot["coef_standardized"])
plt.axvline(0, color="black", linewidth=1)
plt.title("Standardized logistic coefficients")
plt.xlabel("coefficient")
plt.tight_layout()
plt.show()


## 07-5. 상위 Feature Threshold Sweep

상위 feature별로 `feature <= threshold`, `feature >= threshold` 두 방향을 모두 탐색한다. threshold는 train set에서 선택하고, test set에서 다시 평가한다.

운영 관점에서는 `false_remove_rate_nonmicro`가 중요하다. 이 값은 미세스크래치가 아닌 항목을 잘못 제거한 비율이다.

In [ ]:
def threshold_predictions(values: pd.Series, threshold: float, direction: str) -> np.ndarray:
    x = pd.to_numeric(values, errors="coerce")
    pred = np.zeros(len(x), dtype=bool)
    valid = x.notna()
    if direction == "le":
        pred[valid] = x[valid] <= threshold
    elif direction == "ge":
        pred[valid] = x[valid] >= threshold
    else:
        raise ValueError(direction)
    return pred

def threshold_sweep(frame_X: pd.DataFrame, y_true: pd.Series, features: list[str], split_name: str) -> pd.DataFrame:
    rows = []
    quantiles = np.linspace(0.02, 0.98, 97)
    for feature in features:
        x = pd.to_numeric(frame_X[feature], errors="coerce")
        valid_values = x.dropna()
        if valid_values.nunique() < 2:
            continue
        thresholds = np.unique(np.nanquantile(valid_values, quantiles))
        for direction in ["le", "ge"]:
            for threshold in thresholds:
                pred = threshold_predictions(x, threshold, direction)
                metrics = binary_filter_metrics(y_true, pred)
                utility = metrics["micro_remove_recall"] - FALSE_REMOVE_WEIGHT * metrics["false_remove_rate_nonmicro"]
                rows.append({
                    "split": split_name,
                    "feature": feature,
                    "direction": direction,
                    "threshold": float(threshold),
                    "utility": utility,
                    **metrics,
                })
    return pd.DataFrame(rows)

top_features = ranking.head(TOP_K_FEATURES)["feature"].tolist()
print("top features for threshold sweep:")
for i, feat in enumerate(top_features, 1):
    print(f"{i}. {feat}")

train_sweep = threshold_sweep(X_train, y_train, top_features, "train")
sweep_path = RUNS_ROOT / "regression_threshold_sweep_top_features.csv"
train_sweep.to_csv(sweep_path, index=False, encoding="utf-8-sig")
print("saved:", sweep_path)
display(train_sweep.sort_values("utility", ascending=False).head(30))


In [ ]:
def select_best_thresholds(sweep_df: pd.DataFrame) -> pd.DataFrame:
    selected = []
    for feature, part in sweep_df.groupby("feature"):
        eligible = part[part["false_remove_rate_nonmicro"] <= MAX_FALSE_REMOVE_RATE]
        if len(eligible) > 0:
            row = eligible.sort_values(
                ["micro_remove_recall", "precision_micro_when_removed", "utility"],
                ascending=False,
            ).iloc[0].copy()
            row["selection_policy"] = f"best_recall_under_fp<={MAX_FALSE_REMOVE_RATE}"
            selected.append(row)
        row = part.sort_values("utility", ascending=False).iloc[0].copy()
        row["selection_policy"] = f"best_utility_fp_weight={FALSE_REMOVE_WEIGHT}"
        selected.append(row)
    return pd.DataFrame(selected).reset_index(drop=True)

selected_thresholds = select_best_thresholds(train_sweep)
display(selected_thresholds.sort_values(["selection_policy", "utility"], ascending=[True, False]))


In [ ]:
def evaluate_selected_thresholds(selected: pd.DataFrame, X_by_split: dict, y_by_split: dict) -> pd.DataFrame:
    rows = []
    for _, rule in selected.iterrows():
        feature = rule["feature"]
        direction = rule["direction"]
        threshold = float(rule["threshold"])
        for split, X_part in X_by_split.items():
            pred = threshold_predictions(X_part[feature], threshold, direction)
            rows.append({
                "selection_policy": rule["selection_policy"],
                "feature": feature,
                "direction": direction,
                "threshold": threshold,
                "eval_split": split,
                **binary_filter_metrics(y_by_split[split], pred),
            })
    return pd.DataFrame(rows)

threshold_eval = evaluate_selected_thresholds(
    selected_thresholds,
    {"train": X_train, "test": X_test, "full": X},
    {"train": y_train, "test": y_test, "full": y},
)

selected_path = RUNS_ROOT / "regression_threshold_selected_summary.csv"
threshold_eval.to_csv(selected_path, index=False, encoding="utf-8-sig")
print("saved:", selected_path)
display(threshold_eval.sort_values(["selection_policy", "feature", "eval_split"]))


In [ ]:
plt.figure(figsize=(7, 5))
for feature in top_features[:6]:
    part = train_sweep[train_sweep["feature"] == feature]
    best = part.sort_values("utility", ascending=False).iloc[0]
    plt.scatter(part["false_remove_rate_nonmicro"], part["micro_remove_recall"], s=12, alpha=0.25)
    plt.scatter(best["false_remove_rate_nonmicro"], best["micro_remove_recall"], s=60, label=feature)
plt.axvline(MAX_FALSE_REMOVE_RATE, color="red", linestyle="--", linewidth=1, label="FP limit")
plt.xlabel("false remove rate: non-micro incorrectly removed")
plt.ylabel("micro remove recall")
plt.title("Threshold trade-off on train set")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
plot_features = top_features[:min(6, len(top_features))]
fig, axes = plt.subplots(len(plot_features), 1, figsize=(9, 3.2 * len(plot_features)))
if len(plot_features) == 1:
    axes = [axes]

for ax, feature in zip(axes, plot_features):
    x0 = pd.to_numeric(X[feature][y == 0], errors="coerce").dropna()
    x1 = pd.to_numeric(X[feature][y == 1], errors="coerce").dropna()
    ax.hist(x0, bins=40, alpha=0.55, label="keep: non-micro", density=True)
    ax.hist(x1, bins=40, alpha=0.55, label="remove: micro", density=True)
    rules = selected_thresholds[selected_thresholds["feature"] == feature]
    for _, rule in rules.iterrows():
        ax.axvline(rule["threshold"], linestyle="--", linewidth=1.2, label=f"{rule['direction']} {rule['threshold']:.3g}")
    ax.set_title(feature)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 07-6. Logistic Regression Probability Threshold

개별 feature threshold와 별개로, Logistic Regression의 예측 확률 자체에 threshold를 걸 수도 있다. 이 방식은 여러 feature를 동시에 반영하지만, 단일 feature threshold보다 해석성은 떨어진다.

In [ ]:
prob_full = model.predict_proba(X)[:, 1]

def probability_sweep(prob, y_true, split_name):
    rows = []
    for threshold in np.linspace(0.01, 0.99, 99):
        pred = prob >= threshold
        metrics = binary_filter_metrics(y_true, pred, prob)
        utility = metrics["micro_remove_recall"] - FALSE_REMOVE_WEIGHT * metrics["false_remove_rate_nonmicro"]
        rows.append({"split": split_name, "prob_threshold": threshold, "utility": utility, **metrics})
    return pd.DataFrame(rows)

prob_sweep_train = probability_sweep(prob_train, y_train, "train")
prob_sweep_test = probability_sweep(prob_test, y_test, "test")
prob_sweep_full = probability_sweep(prob_full, y, "full")
prob_sweep = pd.concat([prob_sweep_train, prob_sweep_test, prob_sweep_full], ignore_index=True)

eligible = prob_sweep_train[prob_sweep_train["false_remove_rate_nonmicro"] <= MAX_FALSE_REMOVE_RATE]
if len(eligible) > 0:
    selected_prob_threshold = float(eligible.sort_values(["micro_remove_recall", "utility"], ascending=False).iloc[0]["prob_threshold"])
else:
    selected_prob_threshold = float(prob_sweep_train.sort_values("utility", ascending=False).iloc[0]["prob_threshold"])

prob_sweep_path = RUNS_ROOT / "regression_probability_threshold_sweep.csv"
prob_sweep.to_csv(prob_sweep_path, index=False, encoding="utf-8-sig")
print("selected probability threshold:", selected_prob_threshold)
print("saved:", prob_sweep_path)
display(prob_sweep[prob_sweep["prob_threshold"].round(4).eq(round(selected_prob_threshold, 4))])

plt.figure(figsize=(7, 5))
for split, part in prob_sweep.groupby("split"):
    plt.plot(part["false_remove_rate_nonmicro"], part["micro_remove_recall"], marker=".", label=split)
plt.axvline(MAX_FALSE_REMOVE_RATE, color="red", linestyle="--", linewidth=1, label="FP limit")
plt.xlabel("false remove rate: non-micro incorrectly removed")
plt.ylabel("micro remove recall")
plt.title("Model probability threshold trade-off")
plt.legend()
plt.tight_layout()
plt.show()


## 07-7. Row-level Prediction 저장

선택된 threshold가 각 component를 어떻게 판정했는지 CSV로 저장한다. 실제 raw/mask와 다시 매칭해 오제거 사례를 확인할 때 사용한다.

In [ ]:
def safe_rule_name(feature, direction, threshold):
    name = re.sub(r"[^0-9A-Za-z가-힣_]+", "_", feature)
    return f"pred_{name}_{direction}_{threshold:.4g}"

base_cols = [c for c in ["image_path", "component_id", "group", "target_filter"] if c in model_df.columns]
row_pred = model_df[base_cols].copy()
row_pred["model_prob_micro"] = prob_full
row_pred[f"model_pred_prob_ge_{selected_prob_threshold:.2f}"] = (prob_full >= selected_prob_threshold).astype(int)

selected_low_fp_rules = selected_thresholds[selected_thresholds["selection_policy"].str.startswith("best_recall_under_fp")]
for _, rule in selected_low_fp_rules.iterrows():
    feature = rule["feature"]
    direction = rule["direction"]
    threshold = float(rule["threshold"])
    col_name = safe_rule_name(feature, direction, threshold)
    row_pred[col_name] = threshold_predictions(X[feature], threshold, direction).astype(int)

row_pred_path = RUNS_ROOT / "regression_row_predictions.csv"
row_pred.to_csv(row_pred_path, index=False, encoding="utf-8-sig")
print("saved:", row_pred_path)
display(row_pred.head(50))


## 해석 기준

1. 먼저 `regression_feature_ranking.csv`에서 상위 feature가 형상/contrast/noise 중 어디에 몰리는지 본다.
2. `regression_threshold_selected_summary.csv`에서 test split 기준 `false_remove_rate_nonmicro`가 낮은 rule을 우선한다.
3. 같은 false remove rate라면 `micro_remove_recall`이 높은 threshold가 더 좋다.
4. feature 하나의 threshold가 불안정하면 `model_prob_micro` threshold 또는 2개 이상의 rule 조합을 검토한다.

운영 기준은 미세스크래치 제거율을 최대화하는 값이 아니라, 다른 불량 오제거율을 허용 범위 안에 묶은 상태에서 미세스크래치 제거율이 가장 높은 값으로 잡는 것이 안전하다.